# Optional extension — Course review and mini-projects

## Genel tekrar ve mini projeler

**Role in the course:** Beyond the scheduled calendar. Formula review across the course and three computational projects.

This notebook is **not** a scheduled calendar week. Use it for deeper reading, extra examples and practice after the related weekly notebook. Its problems keep the identifier *Module 14 Pn* for the solution collection.

**TR:** Bu not takvimde ayrı bir hafta değildir; ilgili haftadan sonra ek okuma ve alıştırma için kullanılır.

## Contents / İçindekiler

1. [Before you start](#x14-before)
   · [Setup for the interactive graphs (run once)](#x14-setup)
2. [Concepts, demonstrations and worked examples](#x14-concepts) — 0 worked examples, 3 interactive graphs
3. [Problem set — predict, then check](#x14-problems) — 10 problems
4. [Mini-projects](#x14-projects)

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)


<a id="x14-before"></a>

## 1. Before you start / Başlamadan önce

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Recall** the key formulas and concepts from the entire PHY101 semester
2. **Apply** multiple physics concepts in an integrated mini-project
3. **Calibrate** simulation parameters against measurement data
4. **Analyse** data using curve fitting, parameter sweeps, and energy methods
5. **Communicate** physics results through clear plots and numerical analysis

### Joining this lesson / Derse buradan başlayanlar

**Quick recap.** Choose the physical law from the situation: $\sum F=ma$ for acceleration, $E_i+W_{\mathrm{nonconservative}}=E_f$ for energy transfer, and $\sum\vec p_i=\sum\vec p_f$ for a collision with negligible external impulse. A body at rest also needs $\sum\tau=0$. Ideal rolling requires $v=R\omega$ and includes both $\tfrac12mv^2$ and $\tfrac12I\omega^2$. State the axis, initial conditions and loss assumptions before substituting numbers.

**One verification, shared engineering questions.** A 0.50 kg cart starts from rest and receives 10 J of net work on a horizontal track. From $W=\Delta K$, $10=\tfrac12(0.50)v^2$, hence $v=\sqrt{40}=6.32\,\mathrm{m/s}$. A mechanical/mechatronics engineer checks whether friction and the moving load were included in the net work. A software/computer engineer checking a supplied simulation can compare its final energy with 10 J and investigate unit or model errors if the result differs. A smooth animation alone does not establish a correct model.

**TR:** Yeni bir problemde önce başlangıç ve son durumu çiz. Hangi enerji ya da momentum hesabının geçerli olduğunu söyle; kodu anlamak zorunlu değil, fiziksel kontrolü açıklamak önemlidir.

---

### Before we review: choose a law before choosing numbers / Formülden önce fizik

**Core route:** draw the situation → define the start/end or interaction → choose a law → rearrange symbols → substitute SI values → explain the result. Use the formula list as a reference, not a list to memorize without conditions. Coding and numerical fitting are optional verification tools; a clear paper solution and interpretation of a provided plot are sufficient for physics practice.

<table width="100%">
<thead>
<tr>
<th align="left" width="288" scope="col">Clue in the problem</th>
<th align="left" width="472" scope="col">Useful first question / law</th>
</tr>
</thead>
<tbody>
<tr>
<td>Height, spring compression, speed</td>
<td>Where does energy go? $E_i+W_{\rm nonconservative}=E_f$</td>
</tr>
<tr>
<td>Brief collision</td>
<td>Is external impulse negligible? Conserve total momentum.</td>
</tr>
<tr>
<td>Beam at rest</td>
<td>Are both force balance and torque balance satisfied?</td>
</tr>
<tr>
<td>Rolling</td>
<td>Include translation <em>and</em> rotation, with $v=R\omega$.</td>
</tr>
<tr>
<td>Repeating motion</td>
<td>Is the frequency free, damped, or imposed by a driver?</td>
</tr>
</tbody>
</table>

**Algebra bridge:** If $\tfrac12mv^2=E$, multiply by $2/m$ and take $v=\sqrt{\frac{2E}{m}}$. If $mgd\sin\theta=E$, divide by the entire product: $d=\frac{E}{mg\sin\theta}$. If $R=v_xt$, do not substitute the full speed for its horizontal component. Track $+$ and $-$ for velocities; speed itself is nonnegative.

**Five-minute pause:** Explain why momentum can be conserved in a collision even when kinetic energy is not. Then explain why a rolling object stores more energy than a sliding particle of the same mass and speed. **TR:** Hangi büyüklüğün korunduğunu söylemeden işlem yapma. Her formülün geçerli olduğu koşulu yanında tut.

<a id="x14-setup"></a>

## Setup for the interactive graphs (run once) / Kurulum — bir kez çalıştır

Run the cells in this section once per session, then run any **Run the demonstration** cell below. Each demonstration shows a status line under its controls: **Updating…** while the graph is drawn, then the draw time and whether the sliders update live or on release. Nothing here needs to be edited.  
**TR:** Bu bölümdeki hücreleri oturum başına bir kez çalıştır; sonra istediğin gösterimi çalıştır. Kontrollerin altındaki durum satırı, grafiğin ne zaman güncellendiğini gösterir.

In [ ]:
#@title Run once — prepare the physics demonstrations
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from IPython.display import HTML, display, Markdown
import ipywidgets as widgets
from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown, HBox, VBox
from scipy.optimize import curve_fit
from scipy.integrate import solve_ivp

%matplotlib inline

try:
    import google.colab
    IN_COLAB = True
    from matplotlib import rc
    rc('animation', html='jshtml')
except ImportError:
    IN_COLAB = False

plt.rcParams.update({'font.size': 12, 'figure.figsize': (10, 5)})
print("\u2705 All libraries loaded successfully!")

In [ ]:
#@title Run once — prepare the demonstration controls
"""Shared demonstration interface, embedded in every PHY101 notebook.

Only standard ipywidgets, IPython and matplotlib are used, so the notebooks stay
self-contained in Colab and in a local Jupyter. The design goals are:

* A slider change must always produce a visible reaction. The status line under
  the controls says "Updating…" immediately and reports the draw time afterwards.
* The graph is replaced through the same Output-widget route that
  ``ipywidgets.interact`` uses (``clear_output(wait=True)`` followed by a fresh
  display), which is the most widely tested path in Colab and Jupyter. No
  output-capturing context is used: ipykernel 7 dispatches widget messages
  concurrently and IPython's capture object breaks that dispatch.
* Live updates while dragging are switched on when a graph draws quickly and
  switched off (update on release) when it draws slowly, so the kernel never
  falls behind a fast slider.
"""
import functools
import sys
import time
import traceback

import ipywidgets as widgets
from IPython import get_ipython
from IPython.display import HTML, clear_output, display

# Force the inline backend. Otherwise a local kernel may choose a desktop
# backend and block at plt.show(), which looks like a frozen notebook.
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")

_physics_panels = []
_physics_callback_errors = []

# Draw-time thresholds (seconds) for switching live dragging on and off.
PHYSICS_LIVE_ON = 0.12
PHYSICS_LIVE_OFF = 0.25


def physics_frames(frame_count, maximum=60):
    """Sample display frames, retaining both endpoints and all simulation data."""
    count = int(frame_count)
    shown = min(count, maximum)
    if shown <= 1:
        return list(range(shown))
    return [round(index * (count - 1) / (shown - 1)) for index in range(shown)]


def physics_interval(frame_count, interval_ms):
    """Preserve first-to-last playback duration when display frames are sampled."""
    shown = len(physics_frames(frame_count))
    return interval_ms if shown <= 1 else interval_ms * (int(frame_count) - 1) / (shown - 1)


PHYSICS_STYLE = """<style>
.phy101-panel { border: 1px solid #a9b9c9; border-radius: 8px; padding: 8px; background: #fff; }
.phy101-controls { padding: 0 0 2px; box-sizing: border-box; }
.phy101-status { font-size: 12px; color: #4a5a6a; padding: 0 2px 6px; min-height: 18px; }
.phy101-status.busy { color: #b45309; }
.phy101-plot-output img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-plot-output .output_area { overflow: visible; }
.phy101-plot-output table { font-size: 13px; width: 100%; }
.phy101-plot-output .animation { max-width: 100%; }
.phy101-plot-output .animation img { max-width: 100%; height: auto; object-fit: contain; }
.phy101-animation-panel { max-width: 100%; }
.phy101-animation-panel .animation { display: flex; flex-direction: column; }
.phy101-animation-panel .animation img { order: 2; max-width: 100%; height: auto; }
.phy101-animation-panel .anim-controls { order: 1; background: white; color: #172433; padding: 4px; }
@media (max-width: 650px) {
  .phy101-controls, .phy101-plot-output { width: 100% !important; }
}
</style>"""
display(HTML(PHYSICS_STYLE))


def physics_animation_html(animation):
    """A self-contained animation pane with playback controls kept in view."""
    return HTML(PHYSICS_STYLE + '<div class="phy101-animation-panel">' +
                animation.to_jshtml(default_mode="once") + "</div>")


def _physics_controls(items):
    """Lay the controls out as a wrapping toolbar with full-length labels."""
    flat = []
    for item in items:
        if isinstance(item, (widgets.HBox, widgets.VBox)):
            flat.extend(item.children)
        else:
            flat.append(item)
    for control in flat:
        if hasattr(control, "style") and "description_width" in control.style.traits():
            control.style.description_width = "initial"
        control.layout.width = "310px"
        control.layout.flex = "0 1 310px"
        control.layout.max_width = "100%"
        control.layout.min_width = "0"
        control.layout.margin = "2px 6px 2px 0"
        if isinstance(control, widgets.Button):
            control.layout.width = "auto"
            control.layout.flex = "0 0 auto"
    # Repeat the style inside the widget tree: Colab isolates output frames.
    style = widgets.HTML(value=PHYSICS_STYLE, layout=widgets.Layout(display="none"))
    box = widgets.Box([style] + flat, layout=widgets.Layout(
        display="flex", flex_flow="row wrap", align_items="center",
        width="100%", min_width="0", max_width="100%"))
    box.add_class("phy101-controls")
    return box


def _physics_output(output):
    output.layout = widgets.Layout(
        width="100%", min_width="0", max_width="100%",
        height="auto", overflow="visible", margin="0")
    output.add_class("phy101-plot-output")
    return output


def physics_panel(controls, output):
    """Button-driven demos: a compact control toolbar directly above the result."""
    panel = widgets.Box([_physics_controls(controls), _physics_output(output)],
        layout=widgets.Layout(display="flex", flex_flow="column",
                              align_items="stretch", width="100%"))
    panel.add_class("phy101-panel")
    return panel


def physics_show_figure(figure):
    """Display one inline figure and close its pyplot registration afterwards."""
    import matplotlib.pyplot as plt
    display(figure)
    plt.close(figure)


def physics_vector_axes(axes, points):
    """Equal x/y scales and limits covering all arrow endpoints, including sums."""
    import numpy as np
    coordinates = np.asarray(points, dtype=float).reshape(-1, 2)
    span = max(1.0, float(np.max(np.abs(coordinates)))) * 1.22
    axes.set(xlim=(-span, span), ylim=(-span, span), xlabel="x component", ylabel="y component")
    axes.set_aspect("equal", adjustable="box")
    axes.axhline(0, color="#718096", linewidth=0.7)
    axes.axvline(0, color="#718096", linewidth=0.7)
    axes.grid(alpha=0.2)


class PhysicsPanel:
    """Controls, a status line and one Output widget that shows the latest result."""

    def __init__(self, function, controls):
        self.f = function
        self.controls = controls
        self.out = _physics_output(widgets.Output())
        self.status = widgets.HTML(value="")
        self.status.add_class("phy101-status")
        self.seconds = None
        self.live = True
        self.updates = 0
        self.last_outputs = 0
        self.figures = 0
        self.error = None
        visible = []
        for control in controls.values():
            if isinstance(control, widgets.fixed):
                continue
            visible.append(control)
            if hasattr(control, "continuous_update"):
                control.continuous_update = True
            control.observe(self._changed, names="value")
        self.widget = widgets.VBox([_physics_controls(visible), self.status, self.out],
                                   layout=widgets.Layout(width="100%"))
        self.widget.add_class("phy101-panel")
        self.children = self.widget.children
        _physics_panels.append(self)
        self.render()

    # Compatibility with the earlier validation code.
    @property
    def layout(self):
        return self.widget.layout

    def _changed(self, change):
        self.render()

    def _set_live(self, live):
        if live == self.live:
            return
        self.live = live
        for control in self.controls.values():
            if hasattr(control, "continuous_update"):
                control.continuous_update = live

    def render(self):
        self.status.value = "⏳ Updating… / Güncelleniyor…"
        self.status.add_class("busy")
        started = time.perf_counter()
        kwargs = {name: control.value for name, control in self.controls.items()}
        self.error = None
        self.figures = 0
        # Count everything the demonstration shows (figures, HTML, animations, text)
        # by wrapping the display publisher and stdout for this draw only.
        shell = get_ipython()
        publisher = getattr(shell, "display_pub", None) if shell is not None else None
        if publisher is not None:
            original_publish = publisher.publish

            def counting_publish(*args, **kwargs):
                self.figures += 1
                return original_publish(*args, **kwargs)
            publisher.publish = counting_publish
        stdout = sys.stdout
        original_write = stdout.write

        def counting_write(text):
            if text.strip():
                self.figures += 1
            return original_write(text)
        stdout.write = counting_write
        try:
            # The previous result stays visible until the new one arrives.
            with self.out:
                clear_output(wait=True)
                try:
                    result = self.f(**kwargs)
                    from ipywidgets.widgets.interaction import show_inline_matplotlib_plots
                    show_inline_matplotlib_plots()
                    if result is not None:
                        display(result)
                except Exception:
                    self.error = traceback.format_exc()
                    _physics_callback_errors.append((getattr(self.f, "__name__", "callback"), self.error))
                    print(self.error)
        finally:
            if publisher is not None and publisher.__dict__.get("publish") is counting_publish:
                del publisher.publish
            if stdout.__dict__.get("write") is counting_write:
                del stdout.write
        self.last_outputs = self.figures
        self.seconds = time.perf_counter() - started
        self.updates += 1
        if self.seconds > PHYSICS_LIVE_OFF:
            self._set_live(False)
        elif self.seconds < PHYSICS_LIVE_ON:
            self._set_live(True)
        self.status.remove_class("busy")
        mode = ("updates while you drag / sürüklerken güncellenir" if self.live
                else "updates when you release the slider / kaydırıcıyı bırakınca güncellenir")
        self.status.value = (f"✓ Drawn in {self.seconds:.2f} s · {mode}" if self.error is None
                             else "⚠ The demonstration reported an error; see the message below.")


def physics_interactive(function, **controls):
    """Build a panel like ipywidgets.interactive, returning the panel object."""
    @functools.wraps(function)
    def checked(*args, **kwargs):
        return function(*args, **kwargs)

    return PhysicsPanel(checked, controls)


def physics_interact(function=None, **controls):
    """Support both @physics_interact(...) and physics_interact(function, ...)."""
    if function is None:
        return lambda function: physics_interact(function, **controls)
    panel = physics_interactive(function, **controls)
    function.widget = panel.widget
    function.panel = panel
    display(panel.widget)
    return function


<a id="x14-concepts"></a>

## 2. Concepts, demonstrations and worked examples / Konular, gösterimler ve çözümlü örnekler

### Meaning before simulation: make the model explainable

**Optional enrichment; no programming prerequisite or additional exam scope.** Start each project with a labelled physical diagram, the quantity to predict and one sentence for each operation in the model. A plotted curve is useful only when you can say what its axes, slope, area and sign represent.

For a mechanical gripper, identify the forces and slip condition. For a camera-tracked trajectory, distinguish position noise from acceleration inferred by taking slopes twice. For a process-vessel support, distinguish force balance from torque balance. Choose the scenario that interests you; all use the same physical reasoning without requiring disciplinary software.

**Predict before running.** Set one parameter to a simple limiting value, write the expected result and identify which assumptions still hold. Then test a second scenario that changes one assumption rather than just its numbers.

<details><summary>What a useful comparison establishes</summary>

Agreement with a calculation using the same equations checks implementation, not the truth of the physical model. Independent measurements can test the model. A mismatch requires examining measurement uncertainty, neglected interactions and numerical error separately. Explain what observation would make you revise the model.

</details>

**TR:** Benzetimin aynı denklemi yeniden üretmesi modeli deneysel olarak doğrulamaz. Bağımsız veriyle hangi varsayımı test ettiğini belirt.

### Comprehensive Formula Review

Below is a topic reference drawn from the source-module library. The legacy topic-group headings below are not the dated teaching calendar. Use this as a formula sheet together with each model’s assumptions; the actual dated sequence is linked near the start.

### 📊 Kinematics

<table width="100%">
<thead>
<tr>
<th align="left" width="224" scope="col">Equation</th>
<th align="left" width="312" scope="col">Description</th>
<th align="left" width="320" scope="col">Variables</th>
</tr>
</thead>
<tbody>
<tr>
<td>$v = v_0 + at$</td>
<td>Velocity under constant acceleration</td>
<td>$v_0$: initial velocity, $a$: acceleration</td>
</tr>
<tr>
<td>$x = x_0 + v_0 t + \tfrac{1}{2}at^2$</td>
<td>Position under constant acceleration</td>
<td>$x_0$: initial position</td>
</tr>
<tr>
<td>$v^2 = v_0^2 + 2a(x - x_0)$</td>
<td>Velocity-position relation</td>
<td>No time variable</td>
</tr>
<tr>
<td>$x(t) = v_0 \cos\theta\, t$</td>
<td>Projectile horizontal</td>
<td>$\theta$: launch angle</td>
</tr>
<tr>
<td>$y(t) = v_0 \sin\theta\, t - \tfrac{1}{2}g t^2$</td>
<td>Projectile vertical</td>
<td>$g \approx 9.81\,\mathrm{m/s^2}$</td>
</tr>
<tr>
<td>$R = \frac{v_0^2 \sin 2\theta}{g}$</td>
<td>Range (flat ground)</td>
<td>Maximum at $\theta = 45^\circ$</td>
</tr>
</tbody>
</table>

#### Use the formula with its conditions / Formülün koşulları

The projectile rows choose launch position as $(0,0)$ and neglect air resistance. The range row additionally requires landing at the launch height. For a horizontal launch from a table, use the vertical fall time first; do not use the level-ground range formula.

**Small worked example:** A cart moves at $v_0=2.0\,\mathrm{m/s}$ and has constant acceleration $a=3.0\,\mathrm{m/s^2}$ for $t=2.0\,\mathrm s$.
$$v=v_0+at=2.0+3.0(2.0)=8.0\,\mathrm{m/s},$$
$$\Delta x=v_0t+\frac12at^2=2(2)+\frac12(3)(2^2)=4+6=10\,\mathrm m.$$
Check using average velocity: $\bar v=(2+8)/2=5\,\mathrm{m/s}$, so $\Delta x=\bar vt=10\,\mathrm m$. The squared time belongs only to the acceleration term.

**Türkçe:** Önce ivmenin sabit olduğunu kontrol et. $t^2$ ile $t$ farklıdır; bütün ifadeyi tek bir çarpım gibi ele alma. İkinci yöntemle aynı yolu bulmak, cebir ve birim kontrolünü güçlendirir.

#### ⏱️ Checkpoint 1 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A cart starts at $2.0\,\mathrm{m/s}$ and accelerates at $3.0\,\mathrm{m/s^2}$ for $2.0\,\mathrm s$.

**Think — 1 minute:** Choose constant-acceleration equations; predict whether distance is greater than 4 m.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$v=2+3(2)=8\,\mathrm{m/s},\qquad
\Delta x=2(2)+\frac12(3)(2^2)=10\,\mathrm m.$$
At unchanged initial speed it would travel only $4\,\mathrm m$. Positive acceleration adds $6\,\mathrm m$, and the average-velocity check gives $[(2+8)/2](2)=10\,\mathrm m$.

**Türkçe:** Sabit ivme varsayımını söyledikten sonra denklemi kullandık. İlk hızdan gelen yol ile ivmeden gelen yolu ayrı hesaplamak işlem hatasını azaltır. Sonuç beklenen şekilde 4 metreden büyüktür.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### 📊 Forces, Circular Motion & Energy

<table width="100%">
<thead>
<tr>
<th align="left" width="184" scope="col">Equation</th>
<th align="left" width="496" scope="col">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td>$\vec{F}_{\text{net}} = m\vec{a}$</td>
<td>Newton's second law</td>
</tr>
<tr>
<td>$f_s \leq \mu_s N$, $f_k = \mu_k N$</td>
<td>Static and kinetic friction</td>
</tr>
<tr>
<td>$a_c = v^2/r = \omega^2 r$</td>
<td>Centripetal acceleration</td>
</tr>
<tr>
<td>$\tan\theta = v^2/(rg)$</td>
<td>Banked curve (no friction)</td>
</tr>
<tr>
<td>$W = \int \vec{F} \cdot d\vec{r}$</td>
<td>Work done by a force</td>
</tr>
<tr>
<td>$KE = \tfrac{1}{2}mv^2$</td>
<td>Kinetic energy</td>
</tr>
<tr>
<td>$PE_g = mgh$</td>
<td>Gravitational potential energy</td>
</tr>
<tr>
<td>$PE_s = \tfrac{1}{2}kx^2$</td>
<td>Spring potential energy</td>
</tr>
<tr>
<td>$W_{\text{net}} = \Delta KE$</td>
<td>Work-energy theorem</td>
</tr>
<tr>
<td>$E_{\text{mech}} = KE + PE$</td>
<td>Mechanical energy (conserved if no non-conservative forces)</td>
</tr>
</tbody>
</table>

**Unit and sign check / Birim ve işaret kontrolü:** $1\,\mathrm J=1\,\mathrm{N\,m}=1\,\mathrm{kg\,m^2/s^2}$. For a box already sliding $2.0\,\mathrm m$ right, friction of $3.0\,\mathrm N$ left does
$$W_f=Fd\cos180^\circ=(3.0)(2.0)(-1)=-6.0\,\mathrm J.$$
Negative work removes mechanical energy from the moving system. Static friction is bounded by $\mu_sN$; it is not automatically equal to that maximum.

**Türkçe:** İşte kuvvet ile yer değiştirme arasındaki açıyı kullan. Eksi enerji, “toplam enerji negatif olmak zorunda” demek değildir; seçilen kuvvetin enerji aktarım yönünü anlatır.

### 📊 Momentum & Rotational Motion

<table width="100%">
<thead>
<tr>
<th align="left" width="208" scope="col">Equation</th>
<th align="left" width="408" scope="col">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td>$\vec{p} = m\vec{v}$</td>
<td>Linear momentum</td>
</tr>
<tr>
<td>$\vec{J} = \Delta\vec{p} = \vec{F}_{\text{avg}} \Delta t$</td>
<td>Impulse-momentum theorem</td>
</tr>
<tr>
<td>$\tau=rF\sin\theta$, $\sum\tau=I\alpha$</td>
<td>Individual torque; net torque about a fixed axis</td>
</tr>
<tr>
<td>$L = I\omega$</td>
<td>Angular momentum</td>
</tr>
<tr>
<td>$I = \sum m_i r_i^2$ or $\int r^2\,dm$</td>
<td>Moment of inertia</td>
</tr>
<tr>
<td>$KE_{\text{rot}} = \tfrac{1}{2}I\omega^2$</td>
<td>Rotational kinetic energy</td>
</tr>
<tr>
<td>$v_{\text{cm}} = R\omega$</td>
<td>Rolling without slipping</td>
</tr>
</tbody>
</table>

### 📊 Angular Momentum & Oscillations

<table width="100%">
<thead>
<tr>
<th align="left" width="328" scope="col">Equation</th>
<th align="left" width="344" scope="col">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td>$\vec{L} = I\vec{\omega}$</td>
<td>Angular momentum</td>
</tr>
<tr>
<td>$\Delta \vec{L} = 0$ (if no net external torque)</td>
<td>Conservation of angular momentum</td>
</tr>
<tr>
<td>$x(t) = A\cos(\omega t + \phi)$</td>
<td>Simple harmonic motion</td>
</tr>
<tr>
<td>$\omega = \sqrt{\frac{k}{m}}$ (spring), $\omega = \sqrt{\frac{g}{L}}$ (pendulum)</td>
<td>Angular frequency</td>
</tr>
<tr>
<td>$x(t) = A e^{-\gamma t}\cos(\omega' t + \phi)$</td>
<td>Damped oscillation ($\gamma = \frac{b}{2m}$)</td>
</tr>
<tr>
<td>$A(\omega) = \frac{F_0/m}{\sqrt{(\omega_0^2 - \omega^2)^2 + (2\gamma\omega)^2}}$</td>
<td>Forced oscillation amplitude (resonance)</td>
</tr>
</tbody>
</table>

**Conditions / Geçerlilik:** The simple pendulum formula needs small angles. The damped cosine form needs underdamping, with $\omega'=\sqrt{\omega_0^2-\gamma^2}$. Its envelope coefficient is not automatically the displacement at release; initial velocity also determines phase. The forced-response formula describes steady state under a specified harmonic **force**. For three-dimensional rigid-body rotation, $\vec L$ need not be parallel to $\vec\omega$; the scalar $L=I\omega$ used here applies about the stated principal/fixed axis.

**Türkçe:** Benzer görünen sembollerin aynı hareketi anlattığını varsayma. Serbest salınım, sönümlü serbest salınım ve dış kuvvetle sürülen hareketin frekanslarını ayrı tanımla. Formül tablosu koşulların yerine geçmez.

### 📊 Waves & Sound

<table width="100%">
<thead>
<tr>
<th align="left" width="192" scope="col">Equation</th>
<th align="left" width="224" scope="col">Description</th>
</tr>
</thead>
<tbody>
<tr>
<td>$y(x,t) = A\sin(kx - \omega t)$</td>
<td>Travelling wave</td>
</tr>
<tr>
<td>$v = \lambda f = \frac{\omega}{k}$</td>
<td>Wave speed</td>
</tr>
<tr>
<td>$v = \sqrt{\frac{T}{\mu}}$</td>
<td>Speed on a string</td>
</tr>
<tr>
<td>$f_n = \frac{nv}{2L}$</td>
<td>Standing wave frequencies</td>
</tr>
<tr>
<td>$\beta=10\log_{10}\!\left(\frac{I}{I_0}\right)$</td>
<td>Sound intensity in dB</td>
</tr>
</tbody>
</table>

#### ⏱️ Checkpoint 2 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A $1.0\,\mathrm{kg}$ cart moving right at $2.0\,\mathrm{m/s}$ sticks to an identical stationary cart; external impulse is negligible.

**Think — 1 minute:** Predict a final speed between 0 and 2 m/s and decide which quantity is conserved.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$m_1v_1+m_2v_2=(m_1+m_2)v_f
\Rightarrow 1(2)+1(0)=2v_f\Rightarrow v_f=1.0\,\mathrm{m/s}.$$
$$K_i=\tfrac12(1)(2^2)=2.0\,\mathrm J,
\qquad K_f=\tfrac12(2)(1^2)=1.0\,\mathrm J.$$
Momentum remains $2.0\,\mathrm{kg\,m/s}$; the missing $1.0\,\mathrm J$ of mechanical energy becomes internal energy. Sticking is not an elastic collision.

**Türkçe:** Yapışma sırasında sistemi iki araba olarak seçtik. Dış itme ihmal edildiği için momentum korunur; kinetik enerji korunmak zorunda değildir. Enerji kaybolmaz, iç enerjiye aktarılır.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### 🎬 Interactive Formula Explorer

Pick a topic and see an interactive example that uses the key formula.

#### Worked review example A: spring energy becomes motion

A 0.40 kg cart is released from rest by a horizontal spring ($k=100\,\mathrm{N/m}$) compressed 0.060 m. Neglect friction. Predict the speed when the spring reaches its natural length.

**1. Identify the energy conversion.** Initially $U_s=\tfrac12kx^2$ and $K=0$; finally $U_s=0$ and $K=\tfrac12mv^2$.

**2. Cancel the common factor and isolate the square of speed.**

$$\frac12kx^2=\frac12mv^2
\quad\Longrightarrow\quad kx^2=mv^2
\quad\Longrightarrow\quad v^2=\frac{kx^2}{m}.$$

**3. Substitute, then take the positive root for speed.**

$$v=\sqrt{\frac{100(0.060)^2}{0.40}}=\sqrt{0.900}=0.949\,\mathrm{m/s}.$$

**4. Check the energy account.**

$$U_{s,i}=0.180\,\mathrm J,\qquad
K_f=\frac12(0.40)(0.949)^2=0.180\,\mathrm J.$$

**TR:** Formülü seçme nedeni enerji dönüşümüdür. Hızın karesini bulduktan sonra karekök almayı unutma.

#### Worked review example B: a rolling cylinder climbs

A solid cylinder rolls without slipping at 3.0 m/s onto a $30^\circ$ incline. How far along the incline does it travel before stopping? Assume ideal rolling and no energy loss.

**1. Include translation and rotation.**

$$K_i=\frac12mv^2+\frac12I\omega^2.$$

**2. Use the solid-cylinder inertia and the rolling constraint.**

$$I=\frac12mR^2,\qquad\omega=\frac vR,$$
$$K_i=\frac12mv^2+\frac12\left(\frac12mR^2\right)\frac{v^2}{R^2}
=\frac34mv^2.$$

**3. Relate distance along the incline to vertical height.** At the top $h=d\sin30^\circ$. Cancel mass from energy conservation, then divide by the whole product $g\sin30^\circ$:

$$\frac34mv^2=mgd\sin30^\circ
\quad\Longrightarrow\quad d=\frac{3v^2}{4g\sin30^\circ}.$$

**4. Substitute and distinguish the two distances.**

$$d=\frac{3(3.0)^2}{4(9.81)(0.5)}=\frac{27}{19.6}=1.38\,\mathrm m,$$
$$h=d(0.5)=0.688\,\mathrm m.$$

**Interpretation:** At the same starting speed, a frictionless sliding particle would rise only $v^2/(2g)=0.459\,\mathrm{m}$; the cylinder also brings rotational energy. **TR:** Eğik yol uzunluğu ile dikey yüksekliği ayır: $h=d\sin\theta$.


<a id="x14-problems"></a>

## 3. Problem set — predict, then check / Problem seti

Reach the symbolic answer and check a limiting case before opening the **Answer**.


### Core problems (L1) / Temel problemler

Everyone should complete these; they follow the worked examples directly.

#### P1  ·  L1
A projectile is launched at $v_0 = 25\,\mathrm{m/s}$ at $\theta = 35^\circ$ above the horizontal on level ground. Find the range and maximum height. Take $g = 9.81\,\mathrm{m/s^2}$.

**Türkçe — problem:** Bir cisim düz zeminden $v_0=25\,\mathrm{m/s}$ hızla yatayın $35^\circ$ üstüne fırlatılır. Menzili ve en büyük yüksekliği bulun; $g=9.81\,\mathrm{m/s^2}$.

<details><summary>Answer</summary>

$$R = \dfrac{v_0^2\sin 2\theta}{g} = \dfrac{625\sin 70^\circ}{9.81} = 59.9\,\mathrm{m};$$

$$H = \dfrac{(v_0\sin\theta)^2}{2g} = 10.5\,\mathrm{m}.$$

**[CORRECTED]** $R$ previously 58.0 m.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P1 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).

#### P2  ·  L1
A $5.0\,\mathrm{kg}$ block slides down a $30^\circ$ frictionless incline from a height of $2.0\,\mathrm{m}$. Find its speed at the bottom using energy conservation.

**Türkçe — problem:** $5.0\,\mathrm{kg}$ blok, düşey yüksekliği $2.0\,\mathrm m$ olan $30^\circ$’lik sürtünmesiz eğik düzlemden kayar. Enerji korunumu ile alt noktadaki sürati bulun. Başlangıç sürati verilmediğinden durgun bırakılma varsayımını belirtin.

<details><summary>Answer</summary>

$v = 6.26\,\mathrm{m/s}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P2 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).

#### P3  ·  L1
A solid cylinder ($m = 4.0\,\mathrm{kg}$, $R = 0.10\,\mathrm{m}$) rotates at $60\,\mathrm{rad/s}$. Calculate its rotational kinetic energy and angular momentum.

**Türkçe — problem:** $m=4.0\,\mathrm{kg}$ ve $R=0.10\,\mathrm m$ olan dolu silindir $60\,\mathrm{rad/s}$ ile döner. Dönme kinetik enerjisini ve açısal momentumunu bulun; silindirin simetri eksenini kullanın.

<details><summary>Answer</summary>

$KE = 36.0\,\mathrm{J}$; $L = 1.20\,\mathrm{kg\,m^2/s}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P3 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).

#### P4  ·  L1
A spring ($k = 250\,\mathrm{N/m}$) is compressed $0.08\,\mathrm{m}$ by a $0.50\,\mathrm{kg}$ ball. When released, what is the ball's speed?

**Türkçe — problem:** $k=250\,\mathrm{N/m}$ olan yay $0.50\,\mathrm{kg}$ bir top ile $0.08\,\mathrm m$ sıkıştırılmıştır. Serbest bırakıldığında topun sürati kaç olur? Hesap için yayın enerjisinin öteleme kinetik enerjisine dönüştüğü anı ve kayıp varsayımınızı belirtin.

<details><summary>Answer</summary>

$v = 1.79\,\mathrm{m/s}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P4 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).

### Pause and explain / Dur ve açıkla

Before the intermediate problems, explain one core result to a partner.

#### ⏱️ Checkpoint 3 of 3 — Think · Pair · Explain

**Small class activity / Kısa sınıf etkinliği.** A $2.0\,\mathrm{kg}$ solid cylinder rolls without slipping at $2.0\,\mathrm{m/s}$. Compare its total kinetic energy with a nonrotating particle of the same mass and speed.

**Think — 1 minute:** Write both energy terms before calculating; use $I=\tfrac12mR^2$ and $\omega=v/R$.

**Pair — 2 minutes:** Compare your sign, units and first equation. Explain a disagreement before checking the model response.

**Explain — 2 minutes:** Write the physical reason for your answer; the calculation alone is not the explanation.

**Worked model response / Adım adım örnek yanıt**

$$K_{\rm roll}=\frac12mv^2+\frac12\left(\frac12mR^2\right)\frac{v^2}{R^2}
=\frac34mv^2=\frac34(2)(2^2)=6.0\,\mathrm J.$$
The particle has $K=\tfrac12(2)(2^2)=4.0\,\mathrm J$. Equal speed does not imply equal total energy: the cylinder carries another $2.0\,\mathrm J$ in rotation.

**Türkçe:** Yarıçapın karesi sadeleşir, dönme enerjisi kaybolmaz. Yuvarlanmada hem kütle merkezinin ilerlemesi hem eksen etrafındaki dönme vardır. Enerji grafiğinde bu iki katkıyı ayrı göstermeliyiz.

**Your explanation / Açıklaman:** write or discuss your reasoning in words, with an equation or sketch where useful.

### Intermediate problems (L2) / Orta düzey

Combine two ideas from this week.

#### P5  ·  L2
A $2.0\,\mathrm{kg}$ ball moving at $6.0\,\mathrm{m/s}$ collides elastically and head-on with a $4.0\,\mathrm{kg}$ ball at rest. Find the velocities of both balls after the collision.

**Türkçe — problem:** $2.0\,\mathrm{kg}$ top $6.0\,\mathrm{m/s}$ hızla giderken durgun $4.0\,\mathrm{kg}$ topa tam karşıdan esnek çarpar. Çarpışma sonrası iki topun hızlarını yönleriyle bulun.

<details><summary>Answer</summary>

$v_1 = -2.0\,\mathrm{m/s}$; $v_2 = 4.0\,\mathrm{m/s}$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P5 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).

#### P6  ·  L2
A uniform beam of mass $10\,\mathrm{kg}$ and length $3.0\,\mathrm{m}$ is hinged at a wall and supported at its far end by a cable making $45^\circ$ with the beam. A $25\,\mathrm{kg}$ sign hangs from the far end. Find the cable tension and the hinge reaction force components. Take $g = 9.81\,\mathrm{m/s^2}$.

**Türkçe — problem:** Kütlesi $10\,\mathrm{kg}$ ve uzunluğu $3.0\,\mathrm m$ olan düzgün kiriş bir duvara menteşelidir. Uzak ucu, kirişle $45^\circ$ yapan bir kabloyla tutulur; aynı uçtan $25\,\mathrm{kg}$ tabela asılıdır. Kablo gerilmesini ve menteşe tepki kuvvetinin bileşenlerini bulun. $g=9.81\,\mathrm{m/s^2}$ alın.

<details><summary>Answer</summary>

For the usual horizontal-beam arrangement, take right and up as positive. Taking torques about the hinge eliminates the unknown hinge forces:

$$T(3.0\sin45^\circ)=10g(1.5)+25g(3.0)=883\,\mathrm{N\,m}.$$

Divide by the cable's perpendicular lever arm:

$$T=\frac{883}{2.12}=416\,\mathrm N.$$

Now apply force balance in each direction:

$$F_x=T\cos45^\circ=294\,\mathrm N,$$
$$\begin{aligned}
F_y&=(10+25)g-T\sin45^\circ\\
&=343-294\\
&=49.1\,\mathrm N.
\end{aligned}$$

The hinge reaction is rightward and upward with these conventions. **[CORRECTED]** The previous $T=484\,\mathrm N$, $F_x=342\,\mathrm N$ and $F_y=0.98\,\mathrm N$ failed torque balance: that tension gives $1030\,\mathrm{N\,m}$ against the required $883\,\mathrm{N\,m}$.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P6 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).


#### P7  ·  L2
A $0.30\,\mathrm{kg}$ block on a horizontal spring ($k = 75\,\mathrm{N/m}$) has damping $b = 0.60\,\mathrm{N\,s/m}$. It is displaced $0.10\,\mathrm{m}$ and released. Find the number of oscillations before the amplitude drops to $0.01\,\mathrm{m}$.

**Türkçe — problem:** $0.30\,\mathrm{kg}$ blok, $k=75\,\mathrm{N/m}$ yatay yaya bağlıdır ve $b=0.60\,\mathrm{N\,s/m}$ sönüm vardır. $0.10\,\mathrm m$ uzaklaştırılıp bırakılır. Genlik $0.01\,\mathrm m$ olana kadar geçen salınım sayısını bulun; bir çevrimin nasıl sayıldığını belirtin.

<details><summary>Answer</summary>

$\gamma=1.00\,\mathrm{s^{-1}}$,

$$\omega_d=\sqrt{249}=15.8\,\mathrm{rad/s},$$

$T_d=0.398\,\mathrm{s}$. The turning-point decay curve reaches 0.01 m at $t=\ln10=2.30\,\mathrm{s}$, or 5.78 periods. The sixth same-side positive peak after release is the first below 0.01 m. The sinusoidal-envelope bound is slightly later, 2.30 s; both give about 5.8 cycles. If counting extrema on both sides, there are two per period.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P7 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).


#### P8  ·  L2
A string of length $1.0\,\mathrm{m}$ with $\mu = 0.005\,\mathrm{kg/m}$ is under $20\,\mathrm{N}$ of tension. Find the first three standing-wave frequencies and the wavelength of the sound produced by the fundamental in air ($v_\text{sound} = 343\,\mathrm{m/s}$).

**Türkçe — problem:** Uzunluğu $1.0\,\mathrm m$ ve çizgisel yoğunluğu $\mu=0.005\,\mathrm{kg/m}$ olan ip $20\,\mathrm N$ gerilme altındadır. İlk üç duran dalga frekansını ve temel frekansın havada oluşturduğu sesin dalga boyunu bulun; $v_{\rm ses}=343\,\mathrm{m/s}$ alın.

<details><summary>Answer</summary>

$f_1 = 31.6\,\mathrm{Hz}$, $f_2 = 63.2\,\mathrm{Hz}$, $f_3 = 94.9\,\mathrm{Hz}$;

$$\lambda_\text{sound} = 10.9\,\mathrm{m}$$


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P8 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).


### Challenge problems (L3) / İleri düzey

Engineering-style problems with several steps; useful preparation for the exams.

#### P9  ·  L3
A solid sphere ($m = 3.0\,\mathrm{kg}$, $R = 0.08\,\mathrm{m}$) rolls without slipping up a $25^\circ$ incline from an initial speed of $5.0\,\mathrm{m/s}$. (a) How far up the incline does it travel before stopping? (b) A hollow sphere of the same mass and radius is launched at the same speed. How far does it go? (c) Explain the difference using energy methods.

**Türkçe — problem:** $m=3.0\,\mathrm{kg}$ ve $R=0.08\,\mathrm m$ olan dolu küre, $5.0\,\mathrm{m/s}$ başlangıç süratiyle $25^\circ$ eğik düzlemde kaymadan yukarı yuvarlanır. (a) Durmadan önce eğim boyunca aldığı yolu bulun. (b) Aynı kütle, yarıçap ve süratte ince içi boş küre ne kadar gider? (c) Farkı enerjiyle açıklayın.

<details><summary>Answer</summary>

For ideal rolling,

$$d=\frac{v_0^2(1+\beta)}{2g\sin\theta}.$$

**(a)** Solid sphere $\beta=2/5$: $d=4.22\,\mathrm{m}$. **(b)** Thin hollow sphere $\beta=2/3$: $d=5.03\,\mathrm{m}$. **(c)** At the same speed, the hollow sphere has more total kinetic energy: 62.5 J versus 52.5 J for these 3 kg bodies. It therefore climbs farther; equal speed does not mean equal total energy.


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P9 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).


#### P10  ·  L3
A cylindrical flywheel ($m_f = 8.0\,\mathrm{kg}$, $R = 0.15\,\mathrm{m}$) is spinning at $\omega_0 = 300\,\mathrm{rad/s}$ about its symmetry axis. A friction-clutch mechanism couples the flywheel's rim to a launch rail, transferring energy to a $m_p = 0.50\,\mathrm{kg}$ projectile initially at rest. The clutch engages for a brief time during which the flywheel does work on the projectile; assume $35\%$ of the flywheel's initial rotational kinetic energy is converted into translational kinetic energy of the projectile (the rest is lost to friction and heat in the clutch). The projectile then leaves the rail horizontally from the edge of a platform $h = 5.0\,\mathrm{m}$ above level ground.

Take $g = 9.81\,\mathrm{m/s^2}$. Ignore air resistance.

**(a)** Calculate the flywheel's initial rotational kinetic energy. ($I_{\text{cylinder}} = \tfrac{1}{2}m_f R^2$)

**(b)** Find the launch speed $v_0$ of the projectile as it leaves the rail.

**(c)** Determine the projectile's range $R_{\text{proj}}$ (horizontal distance from the platform edge to the landing point).

**(d)** Find the projectile's speed and angle of impact with the ground just before landing.

**(e)** If the flywheel is instead replaced by a solid sphere of the same mass and radius, and the same $35\%$ efficiency applies, by what factor does the launch speed change? Explain.

**Türkçe — problem:** $m_f=8.0\,\mathrm{kg}$ ve $R=0.15\,\mathrm m$ olan silindirik volan, simetri ekseni etrafında $\omega_0=300\,\mathrm{rad/s}$ ile döner. Sürtünmeli kavrama, başlangıç dönme enerjisinin yüzde $35$’ini durgun $m_p=0.50\,\mathrm{kg}$ cismin öteleme enerjisine aktarır; kalan kısım kavramada sürtünme ve ısıya gider. Cisim raydan, yerden $h=5.0\,\mathrm m$ yüksekteki platformun kenarından yatay çıkar. $g=9.81\,\mathrm{m/s^2}$ alın ve hava direncini ihmal edin. (a) $I=\tfrac12m_fR^2$ ile volanın başlangıç dönme enerjisini, (b) çıkış hızını, (c) platform kenarından menzili, (d) çarpmadan hemen önce sürati ve yatayla açısını bulun. (e) Aynı kütle, yarıçap, açısal hız ve verimde volan yerine dolu küre kullanılırsa çıkış hızı hangi katsayıyla değişir? Açıklayın.

<details><summary>Answer</summary>

**(a)**

$$I = \tfrac{1}{2}(8.0)(0.15)^2 = 0.090\,\mathrm{kg\,m^2}.$$

$$KE_{\text{rot}} = \tfrac{1}{2}I\omega_0^2 = \tfrac{1}{2}(0.090)(300)^2 = 4050\,\mathrm{J}.$$

**(b)** Energy transferred:

$$KE_p = 0.35 \times 4050 = 1420\,\mathrm{J}.$$

From $\tfrac{1}{2}m_p v_0^2 = 1420$:

$$v_0 = \sqrt{\frac{2(1420)}{0.50}} = 75.3\,\mathrm{m/s}.$$

**(c)** Projectile motion with horizontal launch from height $h = 5.0\,\mathrm{m}$. Time to fall:

$$h = \tfrac{1}{2}g t^2 \Rightarrow t = \sqrt{\frac{2h}{g}} = \sqrt{\frac{2(5.0)}{9.81}} = 1.01\,\mathrm{s}.$$

Range:

$$R_{\text{proj}} = v_0 t = 75.3 \times 1.01 = 76.0\,\mathrm{m}.$$

**(d)** At impact: $v_x = v_0 = 75.3\,\mathrm{m/s}$,

$$v_y = gt = 9.81 \times 1.01 = 9.91\,\mathrm{m/s}$$

(downward). Speed:

$$v = \sqrt{v_x^2 + v_y^2} = \sqrt{75.3^2 + 9.91^2} = 75.9\,\mathrm{m/s}.$$

Impact angle below horizontal:

$$\theta = \arctan\!\left(\frac{v_y}{v_x}\right) = \arctan\!\left(\frac{9.91}{75.3}\right) = 7.5^\circ.$$

**(e)** Solid sphere:

$$I_s = \tfrac{2}{5}m_f R^2 = 0.072\,\mathrm{kg\,m^2},$$

so

$$KE_s = \tfrac{1}{2}(0.072)(300)^2 = 3240\,\mathrm{J}.$$

Ratio:

$$\frac{KE_s}{KE_c}=\frac{I_s}{I_c}=\frac{2/5}{1/2}=\frac45.$$

Since $v_0 \propto \sqrt{KE}$, the launch speed changes by a factor of

$$\sqrt{\frac45} = \frac{2}{\sqrt5} \approx 0.894.$$

The sphere stores less rotational energy for the same $\omega$ because its moment of inertia is smaller ($2/5$ vs $1/2$ of $mR^2$).


</details>

**Your working / Çözümün:** givens with units → diagram → principle → algebra → substitution → answer and check. Use paper or add a text cell.  
*Full worked solution:* Module 14 P10 in the solutions collection (file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026).


<a id="x14-projects"></a>

## 4. Mini-projects / Mini projeler

### Mini-Projects

Choose **ONE** of the three mini-projects below. Each project integrates multiple physics concepts using a provided model and data. Explain the physics and show your calculations; editing Python is optional.

<table width="100%">
<thead>
<tr>
<th align="left" width="96" scope="col">Project</th>
<th align="left" width="224" scope="col">Topic</th>
<th align="left" width="208" scope="col">Key Physics</th>
<th align="left" width="384" scope="col">Key Skills</th>
</tr>
</thead>
<tbody>
<tr>
<td><strong>Project 1</strong></td>
<td>Air Resistance Projectile</td>
<td>Kinematics + drag force</td>
<td>Compare trajectories, justify a drag estimate</td>
</tr>
<tr>
<td><strong>Project 2</strong></td>
<td>Damped Spring-Mass</td>
<td>Oscillations + damping</td>
<td>Read decay and period from data</td>
</tr>
<tr>
<td><strong>Project 3</strong></td>
<td>Rolling Objects</td>
<td>Rotation + energy</td>
<td>Compare energies and final speeds</td>
</tr>
</tbody>
</table>

For the chosen project, record the model, parameter estimate, units, a prediction, and one limitation in words or on paper. Use the prepared controls or code to check your reasoning. Instructor solutions include representative complete walkthroughs of all three projects; noisy data and open choices mean there is no single exact project answer.

### 🚀 Project 1: Air Resistance Projectile — Parameter Fitting

#### Background

In reality, projectiles experience air drag:

$$\vec{F}_{\text{drag}} = -\tfrac{1}{2} C_d \rho A |\vec{v}|\, \vec{v}$$

where $C_d$ is the drag coefficient, $\rho$ is air density, and $A$ is the cross-sectional area.

We define a combined drag parameter $b=\dfrac{C_d\rho A}{2m}$ so that $\vec{a}_{\text{drag}} = -b |\vec{v}| \vec{v}$.

#### Goal
You have "experimental" data of a projectile's trajectory. Your task is to find the drag coefficient $b$ that best fits the data.

#### Physics steps to report

1. Draw gravity downward and drag opposite the velocity. Write $a_x=-b|v|v_x$, $a_y=-g-b|v|v_y$ and state that $b$ has units m$^{-1}$ here. This $b$ is different from the viscous damping coefficient in Project 2.
2. Compare several values of $b$ with the same measured positions and times. Choose a value that fits both horizontal and vertical motion; a smaller residual sum $\sum[(x_{\rm model}-x_i)^2+(y_{\rm model}-y_i)^2]$ is better.
3. Report a plausible $b$, predicted range/time of flight and a comparison to the no-drag case. Explain what drag changes and why.
4. State a limitation: the data are synthetic/noisy, the model assumes constant drag parameter and no wind, and the supplied time stepping is approximate. A last plotted point above ground is not an exact landing event.

**TR:** Amaç diferansiyel denklem kodlamak değil; sürükleme arttığında yörüngenin nasıl değiştiğini veriyle açıklamaktır.

In [ ]:
#@title Optional numerical check — the algebra is explained above
# === PROJECT 1: Generate synthetic "experimental" data ===
# (This simulates what you would get from a real experiment)

def projectile_with_drag(v0, theta_deg, b, dt=0.01):
    """Simulate projectile with quadratic air drag."""
    g = 9.81
    theta = np.radians(theta_deg)
    vx, vy = v0 * np.cos(theta), v0 * np.sin(theta)
    x, y = 0.0, 0.0
    xs, ys, ts = [x], [y], [0.0]
    t = 0.0
    while y >= 0 or t < 0.01:
        speed = np.sqrt(vx**2 + vy**2)
        ax = -b * speed * vx
        ay = -g - b * speed * vy
        vx += ax * dt
        vy += ay * dt
        x += vx * dt
        y += vy * dt
        t += dt
        if y >= 0:
            xs.append(x); ys.append(y); ts.append(t)
    return np.array(ts), np.array(xs), np.array(ys)

# "True" parameters (unknown to student)
np.random.seed(42)
_v0_true = 30.0
_theta_true = 45.0
_b_true = 0.02

t_true, x_true, y_true = projectile_with_drag(_v0_true, _theta_true, _b_true)

# Sample at regular intervals and add noise
_n_samples = 25
_indices = np.linspace(0, len(t_true)-1, _n_samples, dtype=int)
t_data = t_true[_indices]
x_data = x_true[_indices] + np.random.normal(0, 0.3, _n_samples)
y_data = y_true[_indices] + np.random.normal(0, 0.3, _n_samples)
y_data = np.maximum(y_data, 0)  # no negative heights

# Plot the "experimental" data
fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
ax.plot(x_data, y_data, 'ro', ms=8, label='Experimental data')
# Also show ideal (no drag) for comparison
t_ideal = np.linspace(0, 2*_v0_true*np.sin(np.radians(_theta_true))/9.81, 200)
x_ideal = _v0_true * np.cos(np.radians(_theta_true)) * t_ideal
y_ideal = _v0_true * np.sin(np.radians(_theta_true)) * t_ideal - 0.5*9.81*t_ideal**2
ax.plot(x_ideal, np.maximum(y_ideal, 0), 'g--', lw=1.5, label='No drag (ideal)', alpha=0.7)
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.set_title('Projectile Trajectory: Experimental Data vs Ideal (No Drag)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
plt.show()

print(f"Known: v0 = {_v0_true} m/s, theta = {_theta_true} deg")
print(f"Task: Find the drag coefficient b that best fits the data.")
print(f"Data points: {len(x_data)} measurements")

#### 🎬 Interactive: Drag Parameter Sweep

Use the slider to adjust $b$ and visually match the simulation to the data.

**Compare the same times / Aynı zamanları karşılaştırın.** For $N$ observed positions, the displayed error is

$$\mathrm{RMS}=\sqrt{\frac{1}{N}\sum_{i=1}^N\left[(x_i-x_{\mathrm{model},i})^2+(y_i-y_{\mathrm{model},i})^2\right]}.$$

Squaring removes the signs, adding combines the two coordinate errors, averaging gives a mean squared distance, and the square root returns metres. Every trial must predict every observation time. If a trial lands too early, the demonstration reports that limitation instead of extending its final point silently. Türkçe: farklı denemelerde farklı sayıda veriyi kullanırsak hata değerlerini adil biçimde karşılaştıramayız. “RMS yok” uyarısı grafiğin bozulması değil, modelin ölçüm zamanlarını kapsamamasıdır.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
@physics_interact(b=FloatSlider(min=0.0, max=0.1, step=0.002, value=0.0, description='Drag b:',
                         style={'description_width': 'initial'}, readout_format='.3f'))
def sweep_drag(b):
    t_sim, x_sim, y_sim = projectile_with_drag(_v0_true, _theta_true, b)

    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")
    ax.plot(x_data, y_data, 'ro', ms=8, label='Experimental data', zorder=5)
    ax.plot(x_sim, y_sim, 'b-', lw=2.5, label=f'Simulation (b={b:.3f})')
    ax.plot(x_ideal, np.maximum(y_ideal, 0), 'g--', lw=1, alpha=0.5, label='No drag')

    # A trial that lands before the last observation cannot be compared at all timestamps.
    # np.interp would otherwise silently extend the last point as a constant.
    covered = t_data <= t_sim[-1] + 1e-12
    if covered.all():
        x_interp = np.interp(t_data, t_sim, x_sim)
        y_interp = np.interp(t_data, t_sim, y_sim)
        residual = np.sqrt(np.mean((x_data - x_interp)**2 + (y_data - y_interp)**2))
        fit_label = f'RMS position error = {residual:.2f} m (all {len(t_data)} observations)'
    else:
        fit_label = f'RMS unavailable: trial lands before {(~covered).sum()} observations'

    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
    ax.set_title(f'Drag Parameter Sweep: b = {b:.3f}\n{fit_label}')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
    plt.show()

### 🌀 Project 2: Damped Spring-Mass — Finding Damping from Data

#### Background

A damped harmonic oscillator follows:

$$m\ddot{x} + b\dot{x} + kx = 0$$

The solution (underdamped case) is:

$$x(t) = A_0 e^{-\gamma t} \cos(\omega_d t + \phi)$$

where $\gamma = \frac{b}{2m}$ is the damping rate and $\omega_d = \sqrt{\omega_0^2 - \gamma^2}$.

#### Goal
You have displacement-vs-time data from a damped oscillator. Find the spring constant $k$ and damping coefficient $b$.

#### Physics steps to report

1. Read the period from successive peaks. Estimate $\omega_d=2\pi/T_d$.
2. Compare two same-side peak magnitudes $x_1,x_2$, separated by $\Delta t$. Take a logarithm before dividing:

$$\frac{x_2}{x_1}=e^{-\gamma\Delta t}
\quad\Longrightarrow\quad\ln\!\left(\frac{x_2}{x_1}\right)=-\gamma\Delta t,$$
$$\gamma=\frac{\ln(x_1/x_2)}{\Delta t}.$$

3. Rearrange the parameter definitions: $b=2m\gamma$ and $k=m(\omega_d^2+\gamma^2)$. Fit phase as well as decay if the first observation is not a turning point; the data use $\phi\ne0$, so $A_0$ is not exactly $x(0)$.
4. Compare the model to the points, give a predicted displacement at a chosen time, and describe uncertainty. Near the end, noise can be larger than the remaining oscillation; do not estimate damping from those tiny peaks alone.

**TR:** İki tepe arasındaki zaman periyodu, tepe boylarının oranı sönümü verir. Önce bu iki fiziksel bilgiyi çıkar; sonra $k$ ve $b$'yi hesapla.

In [ ]:
#@title Optional numerical check — the algebra is explained above
# === PROJECT 2: Generate synthetic "experimental" data ===

np.random.seed(123)
_m_true = 0.5     # kg
_k_true = 20.0    # N/m
_b_true_p2 = 0.8  # Ns/m
_A0_true = 0.15   # m
_phi_true = 0.3   # rad

_gamma_true = _b_true_p2 / (2 * _m_true)
_omega0_true = np.sqrt(_k_true / _m_true)
_omegad_true = np.sqrt(_omega0_true**2 - _gamma_true**2)

t_osc = np.linspace(0, 8, 400)
x_osc_clean = _A0_true * np.exp(-_gamma_true * t_osc) * np.cos(_omegad_true * t_osc + _phi_true)
x_osc_data = x_osc_clean + np.random.normal(0, 0.003, len(t_osc))

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
ax.plot(t_osc, x_osc_data, 'b.', ms=2, alpha=0.6, label='Measured data')
ax.set_xlabel('Time (s)'); ax.set_ylabel('Displacement (m)')
ax.set_title('Damped Oscillation: Experimental Measurement')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
plt.show()

print(f"Known: m = {_m_true} kg")
print(f"Task: Find k (spring constant) and b (damping coefficient).")
print(f"Data: {len(t_osc)} time-displacement measurements")

#### 🎬 Interactive: Damping Curve Fitting

Manually adjust the parameters to match the data.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
@physics_interact(A0=FloatSlider(min=0.05, max=0.3, step=0.005, value=0.15, description='A0 (m)'),
          gamma=FloatSlider(min=0.0, max=3.0, step=0.05, value=0.5, description='gamma (1/s)'),
          omega_d=FloatSlider(min=1.0, max=10.0, step=0.1, value=6.0, description='omega_d (rad/s)'),
          phi=FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description='phi (rad)'))
def fit_damped(A0, gamma, omega_d, phi):
    x_fit = A0 * np.exp(-gamma * t_osc) * np.cos(omega_d * t_osc + phi)
    residual = np.sqrt(np.mean((x_osc_data - x_fit)**2))

    fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
    ax.plot(t_osc, x_osc_data, 'b.', ms=2, alpha=0.5, label='Data')
    ax.plot(t_osc, x_fit, 'r-', lw=2, label='Fit')
    ax.plot(t_osc, A0*np.exp(-gamma*t_osc), 'g--', lw=1, alpha=0.7, label='Envelope')
    ax.plot(t_osc, -A0*np.exp(-gamma*t_osc), 'g--', lw=1, alpha=0.7)
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Displacement (m)')
    ax.set_title(f'Manual Fit | RMS Error = {residual:.5f} m')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)
    plt.show()

    # Infer physical parameters
    m = _m_true
    b_infer = 2 * m * gamma
    omega0_infer = np.sqrt(omega_d**2 + gamma**2)
    k_infer = m * omega0_infer**2
    print(f"Inferred: k = {k_infer:.2f} N/m, b = {b_infer:.3f} Ns/m")

### 🎡 Project 3: Rolling Objects — Energy Analysis

#### Background

When an object rolls down a ramp without slipping, both translational and rotational kinetic energy are involved:

$$mgh = \tfrac{1}{2}mv^2 + \tfrac{1}{2}I\omega^2$$

For an object with $I = cmR^2$:

$$v = \sqrt{\frac{2gh}{1 + c}}$$

<table width="100%">
<thead>
<tr>
<th align="left" width="144" scope="col">Shape</th>
<th align="left" width="112" scope="col">$c = I/(mR^2)$</th>
</tr>
</thead>
<tbody>
<tr>
<td>Solid sphere</td>
<td>$\dfrac25$</td>
</tr>
<tr>
<td>Solid cylinder</td>
<td>$\dfrac12$</td>
</tr>
<tr>
<td>Hollow sphere</td>
<td>$\dfrac23$</td>
</tr>
<tr>
<td>Hollow cylinder</td>
<td>$1$</td>
</tr>
</tbody>
</table>

#### Goal
Analyse the rolling motion of different shapes, including the effect of energy losses.

#### Physics steps to report

1. Use $I=cmR^2$ and $\omega=v/R$ to show $K=\tfrac12mv^2(1+c)$.
2. For the same ramp, compare all four shapes using the ideal final speed and time. Explain the ordering with energy sharing.
3. The supplied loss model treats an effective resistance as $F_r=\mu_rmg\cos\theta$, giving energy loss $W_r=\mu_rmg\cos\theta\,L_{\rm ramp}$ and
$$v^2=\frac{2gh(1-\mu_r\cot\theta)}{1+c}.$$
Compare an ideal case and a small nonzero $\mu_r$ case; check that $K+U+E_{\rm lost}$ stays constant.
4. If a final speed is measured, first divide the squared-speed equation by $2gh/(1+c)$, then isolate the resistance coefficient:

$$\frac{(1+c)v^2}{2gh}=1-\mu_r\cot\theta,$$
$$\mu_r=\left[1-\frac{(1+c)v^2}{2gh}\right]\tan\theta.$$

This is an **effective rolling-resistance model**, not a measurement of the static friction coefficient. Pure static friction can enforce rolling without energy loss.

**TR:** Kayıp enerjiyi yok sayma; ısıya giden kısmı bilançoya ekle. Kaymadan yuvarlanma için gerekli statik sürtünme ile enerji kaybettiren yuvarlanma direnci farklı kavramlardır.

In [ ]:
#@title Run the demonstration — predict, adjust, observe
# === PROJECT 3: Setup and Visualization ===

def rolling_simulation(h, angle_deg, c, mu_r=0.0, m=1.0, R=0.05):
    """Simulate an object rolling down a ramp.
    c: I/(mR^2) shape factor
    mu_r: rolling friction coefficient (energy loss)
    Returns: times, positions along ramp, speeds, energies
    """
    g = 9.81
    theta = np.radians(angle_deg)
    L_ramp = h / np.sin(theta)  # ramp length

    # Acceleration along ramp: a = g*sin(theta)/(1+c) - mu_r*g*cos(theta)/(1+c)
    a = g * (np.sin(theta) - mu_r * np.cos(theta)) / (1 + c)
    if a <= 0:
        return np.array([0]), np.array([0]), np.array([0]), {}

    t_end = np.sqrt(2 * L_ramp / a) if a > 0 else 10
    t = np.linspace(0, t_end, 300)
    s = 0.5 * a * t**2
    v = a * t

    # Retain the analytically known final time; avoid dropping it through roundoff.
    s = np.minimum(s, L_ramp)
    s[-1] = L_ramp

    height = np.maximum(0.0, h - s * np.sin(theta))
    KE_trans = 0.5 * m * v**2
    KE_rot = 0.5 * c * m * v**2  # since I*omega^2 = c*m*R^2*(v/R)^2 = c*m*v^2
    PE = m * g * height
    E_total = KE_trans + KE_rot + PE
    E_lost = mu_r * m * g * np.cos(theta) * s  # work against the resistance force

    energies = {'KE_trans': KE_trans, 'KE_rot': KE_rot, 'PE': PE,
                'E_total': E_total, 'E_lost': E_lost}
    return t, s, v, energies

# Interactive demo
@physics_interact(shape=Dropdown(options=['Solid sphere (c=0.4)', 'Solid cylinder (c=0.5)',
                                   'Hollow sphere (c=0.667)', 'Hollow cylinder (c=1.0)'],
                          value='Solid sphere (c=0.4)', description='Shape:'),
          h=FloatSlider(min=0.5, max=5.0, step=0.25, value=2.0, description='Height (m)'),
          angle=FloatSlider(min=10, max=60, step=5, value=30, description='Angle (deg)'),
          mu_r=FloatSlider(min=0.0, max=0.1, step=0.005, value=0.0, description='Resistance mu_r:',
                           readout_format='.3f'))
def show_rolling(shape, h, angle, mu_r):
    c_map = {'Solid sphere (c=0.4)': 2/5, 'Solid cylinder (c=0.5)': 1/2,
             'Hollow sphere (c=0.667)': 2/3, 'Hollow cylinder (c=1.0)': 1.0}
    c = c_map[shape]

    t, s, v, E = rolling_simulation(h, angle, c, mu_r)
    if len(t) < 2:
        print("Object cannot roll (friction too high). Reduce mu_r or increase angle.")
        return

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), layout="constrained")

    # Speed vs time
    v_ideal = np.sqrt(2 * 9.81 * h / (1 + c))
    ax1.plot(t, v, 'b-', lw=2.5, label='Speed')
    ax1.axhline(v_ideal, color='g', ls='--', alpha=0.7, label=f'v_max (no friction) = {v_ideal:.2f} m/s')
    ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Speed (m/s)')
    ax1.set_title(f'{shape}\nRolling down a ramp'); ax1.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    # Energy breakdown
    ax2.fill_between(t, 0, E['KE_trans'], alpha=0.4, color='red', label='KE (trans)')
    ax2.fill_between(t, E['KE_trans'], E['KE_trans']+E['KE_rot'], alpha=0.4, color='orange', label='KE (rot)')
    ax2.fill_between(t, E['KE_trans']+E['KE_rot'], E['KE_trans']+E['KE_rot']+E['PE'],
                     alpha=0.4, color='blue', label='PE')
    if mu_r > 0:
        ax2.fill_between(t, E['KE_trans']+E['KE_rot']+E['PE'],
                         E['KE_trans']+E['KE_rot']+E['PE']+E['E_lost'],
                         alpha=0.4, color='gray', label='Lost to friction')
    ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Energy (J)')
    ax2.set_title('Energy Breakdown'); ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9)

    plt.show()
    print(f"Final speed: {v[-1]:.3f} m/s | Time to bottom: {t[-1]:.3f} s")

## Solutions / Çözümler

Complete worked solutions: Module 14 (Review and projects), file `Week_14_Python_Solutions.ipynb`, opens 18 December 2026 on the course page.

[Course page / Ders sayfası](https://arifsolmaz.github.io/courses/fall/phy101/web/PHY101_Course_Dashboard.html)